# WGDSA in a Multigroup Source--Detector Problem

This tutorial compares source iteration with and without within-group diffusion synthetic acceleration (WGDSA) in a heterogeneous two-dimensional shielding problem. A compact volumetric source and an energy-dependent detector are separated by a strongly scattering moderator.

**Audience:** Users applying WGDSA to practical multigroup fixed-source calculations.

**Prerequisites:** Groupsets, volumetric sources, and volume postprocessors.

## Reduced multigroup model

The geometry follows a source--detector configuration: a 2 cm square source is centered near the lower-left corner and a separate 2 cm square detector is near the upper-right corner. The HDPE, Cf-252, and He-3 data were reduced from the 69-group WIMS tutorial library to 12 groups. Total and absorption cross sections use an unweighted fine-group average, while transfer terms are averaged over source groups and summed over destination groups. This makes the data suitable for demonstrating solver behavior, but not for design analysis. The source spectrum is a similarly collapsed Cf-252 Watt spectrum.

The mesh uses a 2-by-2 KBA decomposition, so run the generated script with four MPI processes. Sixteen directions keep the comparison inexpensive while retaining a genuine parallel transport sweep.

In [ ]:
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.mesh import KBAGraphPartitioner, OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
num_groups = 12
source_spectrum = [
    8.70673882298e-1, 1.27603333571e-1, 1.69874391340e-3,
    2.39751217396e-5, 6.37406413603e-8, 8.61581497551e-10,
    1.06916189569e-10, 2.94232444417e-10, 5.81630205686e-11,
    2.77856961803e-11, 4.45769378270e-12, 1.77083905175e-12,
]

## Mesh and materials

Block 0 is the moderator, block 1 is the detector, and block 2 is the source region. Logical volumes make the regions independent of external mesh files or command-line parameters.

In [ ]:
def load_xs(filename):
    xs = MultiGroupXS()
    xs.LoadFromOpenSn(str(tutorial_dir / filename))
    return xs


def make_mesh():
    nodes = [0.5 * i for i in range(41)]
    partitioner = KBAGraphPartitioner(
        nx=2, ny=2, xcuts=[10.0], ycuts=[10.0]
    )
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes], partitioner=partitioner
    ).Execute()
    mesh.SetOrthogonalBoundaries()
    mesh.SetUniformBlockID(0)

    detector = RPPLogicalVolume(
        xmin=14.0, xmax=16.0, ymin=14.0, ymax=16.0, infz=True
    )
    source = RPPLogicalVolume(
        xmin=2.0, xmax=4.0, ymin=2.0, ymax=4.0, infz=True
    )
    mesh.SetBlockIDFromLogicalVolume(detector, 1, True)
    mesh.SetBlockIDFromLogicalVolume(source, 2, True)
    return mesh

## Build one solver with optional WGDSA

Each energy group has its own groupset. This preserves the fast-to-thermal downscatter ordering while allowing WGDSA to target the slowly converging spatial error within every group. Both cases use classic Richardson source iteration so the effect of the diffusion correction is visible.

The reported wall time includes initialization, diffusion-system construction when enabled, and transport execution. The maximum time over all ranks is used.

In [ ]:
def solve(use_wgdsa):
    moderator_xs = load_xs("wgdsa_multigroup_moderator.cxs")
    source_xs = load_xs("wgdsa_multigroup_source.cxs")
    detector_xs = load_xs("wgdsa_multigroup_detector.cxs")

    groupset_options = {
        "angular_quadrature": GLCProductQuadrature2DXY(
            n_polar=2, n_azimuthal=16, scattering_order=0
        ),
        "inner_linear_method": "classic_richardson",
        "l_abs_tol": 1.0e-8,
        "l_max_its": 1000,
    }
    if use_wgdsa:
        groupset_options.update(
            {
                "apply_wgdsa": True,
                "wgdsa_l_abs_tol": 1.0e-8,
                "wgdsa_l_max_its": 200,
                "wgdsa_solver_policy": "auto",
            }
        )

    groupsets = [
        dict(groupset_options, groups_from_to=(group, group))
        for group in range(num_groups)
    ]
    problem = DiscreteOrdinatesProblem(
        mesh=make_mesh(),
        num_groups=num_groups,
        groupsets=groupsets,
        xs_map=[
            {"block_ids": [0], "xs": moderator_xs},
            {"block_ids": [1], "xs": detector_xs},
            {"block_ids": [2], "xs": source_xs},
        ],
        volumetric_sources=[
            VolumetricSource(block_ids=[2], group_strength=source_spectrum)
        ],
        boundary_conditions=[
            {"name": "xmin", "type": "vacuum"},
            {"name": "xmax", "type": "vacuum"},
            {"name": "ymin", "type": "vacuum"},
            {"name": "ymax", "type": "vacuum"},
        ],
        options={
            "max_ags_iterations": 20,
            "ags_tolerance": 1.0e-8,
            "ags_convergence_check": "pointwise",
            "verbose_inner_iterations": False,
        },
    )

    solver = SteadyStateSourceSolver(problem=problem)
    comm.Barrier()
    start = MPI.Wtime()
    solver.Initialize()
    solver.Execute()
    comm.Barrier()
    elapsed = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)

    detector_integral = VolumePostprocessor(
        problem=problem, value_type="integral", block_ids=[1]
    )
    detector_integral.Execute()
    detector_flux = detector_integral.GetValue()[0]
    response = sum(
        sigma_a * flux
        for sigma_a, flux in zip(detector_xs.sigma_a, detector_flux)
    )
    return response, elapsed

## Compare the response and runtime

WGDSA changes the convergence path, not the converged detector response. The response comparison is therefore the correctness check. Runtime is printed rather than asserted because it depends on the machine and MPI configuration; the scattering-dominated case should normally show a clear reduction on four ranks.

In [ ]:
unaccelerated_response, unaccelerated_time = solve(False)
accelerated_response, accelerated_time = solve(True)
relative_difference = abs(
    unaccelerated_response - accelerated_response
) / abs(unaccelerated_response)
speedup = unaccelerated_time / accelerated_time

if rank == 0:
    print(f"Unaccelerated detector response={unaccelerated_response:.12e}")
    print(f"WGDSA detector response={accelerated_response:.12e}")
    print(f"Detector response relative difference={relative_difference:.12e}")
    print(f"Unaccelerated wall time (s)={unaccelerated_time:.6f}")
    print(f"WGDSA wall time (s)={accelerated_time:.6f}")
    print(f"WGDSA speedup={speedup:.6f}")

assert relative_difference < 1.0e-6

if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()